In [1]:
import Bio
import os, glob
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa # Helper to check for amino acids
import warnings

# Suppress PDBConstructionWarning, which is common with PDB files
from Bio.PDB.PDBExceptions import PDBConstructionWarning
warnings.simplefilter('ignore', PDBConstructionWarning)

In [2]:
def count_residues_biopython(pdb_file_path):
    """
    Counts the number of standard residues (amino acids/nucleotides) in each chain of a PDB file.

    Args:
        pdb_file_path (str): The full path to the PDB file.

    Returns:
        dict: A dictionary where keys are chain IDs and values are the residue counts.
              Returns an empty dictionary if the file cannot be parsed.
    """
    try:
        parser = PDBParser()
        structure = parser.get_structure("protein", pdb_file_path)
    except Exception as e:
        print(f"Error parsing PDB file: {e}")
        return {}

    residue_counts = {}

    for model in structure:
        for chain in model:
            # We filter out heteroatoms like water (HOH) or ligands.
            # A residue's ID is a tuple: (hetero_flag, sequence_identifier, insertion_code)
            # A standard residue will have a blank hetero_flag.
            # Example: (' ', 123, 'A') -> standard | ('H_W', 456, ' ') -> heteroatom
            
            # Using a list comprehension to count standard residues
            count = len([res for res in chain if res.id[0] == ' '])
            
            residue_counts[chain.id] = count

    return residue_counts

In [14]:
storage = 'alpha_proteo_targets'
pdbid = '3di3'

In [15]:
variants = {
    'raw': os.path.join(storage, pdbid+'_cropped.pdb'),
    'fixed': os.path.join(storage, pdbid+'_cropped_fixed.pdb'),
    'repacked': os.path.join(storage, pdbid+'_repacked.pdb'),
}

In [16]:
for k, v in variants.items():
    if os.path.exists(v):
        residue_counts = count_residues_biopython(v)
        print(f'{pdbid} {k} has {len(residue_counts)} chains')
        for chain_id, count in residue_counts.items():
            print(f'{pdbid} {k} chain {chain_id} has {count} residues')


3di3 raw has 1 chains
3di3 raw chain B has 193 residues
3di3 fixed has 1 chains
3di3 fixed chain A has 193 residues
3di3 repacked has 1 chains
3di3 repacked chain B has 193 residues
